
# SHA-256 Geometry-Only Reverse Game Notebook

This notebook builds a **single-block** SHA-256 reverse game.

## Design boundary

The notebook enforces a hard split between:

1. **Forward sealing**
   - computes the true SHA-256 block trace internally,
   - derives a **geometry-only package**,
   - stores only **sealed commitments** for the message-schedule words,
   - discards direct transport values from the exported game state.

2. **Reverse game**
   - starts from the **hash only**,
   - walks backward one round at a time,
   - asks you for the exact next missing transport word `W[t]`,
   - checks your guess against a sealed commitment,
   - moves backward only when the guess is correct.

## Forbidden in the visible game state

The geometry package shown to the player does **not** expose:

- message bytes,
- padded block bytes,
- message words `W[0..15]`,
- expanded schedule words `W[16..63]`,
- per-round state words `a..h`,
- direct `T1[t]` / `T2[t]` word values.

## Allowed in the visible game state

The package **does** expose only side / geometry observables such as:

- final digest,
- NOP-backbone comparison,
- Hamming weights,
- carry counts / carry bits,
- differential-channel Hamming weights,
- sealed commitments for answer checking.

The reverse engine itself is exact SHA algebra:
it uses the hash-derived final working state and your guessed `W[t]` values to step backward.


In [1]:

# Optional setup cell.
# No third-party packages are required for this notebook.


In [2]:

from __future__ import annotations

from dataclasses import dataclass, field
import hashlib
import os
import struct
from typing import Dict, List, Tuple

MASK32 = 0xFFFFFFFF

H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

def u32(x: int) -> int:
    return x & MASK32

def rotr(x: int, n: int) -> int:
    x &= MASK32
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (~x & z)) & MASK32

def maj(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (x & z) ^ (y & z)) & MASK32

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def hw(x: int) -> int:
    return (x & MASK32).bit_count()

def pad_single_block(msg: bytes) -> bytes:
    if len(msg) > 55:
        raise ValueError("This notebook is restricted to single-block messages of length <= 55 bytes.")
    bit_len = len(msg) * 8
    out = msg + b"\x80"
    while len(out) % 64 != 56:
        out += b"\x00"
    out += struct.pack(">Q", bit_len)
    assert len(out) == 64
    return out

def words_from_block(block: bytes) -> List[int]:
    return list(struct.unpack(">16I", block))

def expand_schedule(w16: List[int]) -> List[int]:
    if len(w16) != 16:
        raise ValueError("Need 16 initial words.")
    W = list(w16)
    for t in range(16, 64):
        W.append(u32(sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16]))
    return W

def hashlib_sha256_hex(msg: bytes) -> str:
    return hashlib.sha256(msg).hexdigest()

def commitment(secret_salt: bytes, label: str, value: int) -> str:
    payload = secret_salt + label.encode("utf-8") + struct.pack(">I", value & MASK32)
    return hashlib.sha256(payload).hexdigest()


In [3]:

@dataclass
class RoundTrace:
    t: int
    a: int
    b: int
    c: int
    d: int
    e: int
    f: int
    g: int
    h: int
    Wt: int
    T1: int
    T2: int
    delta: int
    free: int
    t2_carry: int
    t1_carry_count: int

def compress_single_block_with_trace(msg: bytes) -> Dict[str, object]:
    block = pad_single_block(msg)
    w16 = words_from_block(block)
    W = expand_schedule(w16)

    a, b, c, d, e, f, g, h = H0
    traces: List[RoundTrace] = []

    for t in range(64):
        s1 = Sigma1(e)
        chv = ch(e, f, g)
        sum_t1 = h + s1 + chv + K[t] + W[t]
        T1 = u32(sum_t1)
        T2_sum = Sigma0(a) + maj(a, b, c)
        T2 = u32(T2_sum)

        traces.append(
            RoundTrace(
                t=t, a=a, b=b, c=c, d=d, e=e, f=f, g=g, h=h,
                Wt=W[t], T1=T1, T2=T2,
                delta=u32(T2 - d),
                free=u32(h + W[t]),
                t2_carry=1 if (T2_sum >> 32) else 0,
                t1_carry_count=(sum_t1 >> 32),
            )
        )

        new_a = u32(T1 + T2)
        new_e = u32(d + T1)
        a, b, c, d, e, f, g, h = new_a, a, b, c, new_e, e, f, g

    working_final = [a, b, c, d, e, f, g, h]
    digest_words = [u32(H0[i] + working_final[i]) for i in range(8)]
    digest_hex = "".join(f"{x:08x}" for x in digest_words)
    assert digest_hex == hashlib_sha256_hex(msg)

    return {
        "block": block,
        "w16": w16,
        "W": W,
        "traces": traces,
        "working_final": working_final,
        "digest_words": digest_words,
        "digest_hex": digest_hex,
    }

def nop_backbone_trace() -> Dict[str, object]:
    a, b, c, d, e, f, g, h = H0
    traces = []
    for t in range(64):
        sum_t1 = h + Sigma1(e) + ch(e, f, g) + K[t]
        T1 = u32(sum_t1)
        T2_sum = Sigma0(a) + maj(a, b, c)
        T2 = u32(T2_sum)
        traces.append(
            RoundTrace(
                t=t, a=a, b=b, c=c, d=d, e=e, f=f, g=g, h=h,
                Wt=0, T1=T1, T2=T2,
                delta=u32(T2 - d),
                free=h,
                t2_carry=1 if (T2_sum >> 32) else 0,
                t1_carry_count=(sum_t1 >> 32),
            )
        )
        new_a = u32(T1 + T2)
        new_e = u32(d + T1)
        a, b, c, d, e, f, g, h = new_a, a, b, c, new_e, e, f, g
    return {
        "working_final": [a, b, c, d, e, f, g, h],
        "traces": traces,
    }

NOP = nop_backbone_trace()


In [4]:

@dataclass
class GeometryPackage:
    digest_hex: str
    digest_words: List[int]
    final_state_word_hw: List[int]
    glass_key_word_hw: List[int]
    t2_carry_bits: str
    t1_carry_histogram: Dict[int, int]
    delta_hw: List[int]
    free_hw: List[int]
    nop_delta_hw: List[int]
    round_count: int = 64

    def summary(self) -> None:
        print("digest_hex =", self.digest_hex)
        print("final_state_word_hw =", self.final_state_word_hw)
        print("glass_key_word_hw  =", self.glass_key_word_hw)
        print("t2_carry_bits      =", self.t2_carry_bits)
        print("t1_carry_histogram =", self.t1_carry_histogram)
        print("delta_hw[:8]       =", self.delta_hw[:8], "...")
        print("free_hw[:8]        =", self.free_hw[:8], "...")
        print("nop_delta_hw[:8]   =", self.nop_delta_hw[:8], "...")

@dataclass
class SealedOracle:
    digest_hex: str
    secret_salt: bytes
    word_commitments: Dict[int, str]
    free_commitments: Dict[int, str]
    schedule_commitment: str

    def check_word(self, t: int, value: int) -> bool:
        return self.word_commitments[t] == commitment(self.secret_salt, f"W[{t}]", value)

    def check_free(self, t: int, value: int) -> bool:
        return self.free_commitments[t] == commitment(self.secret_salt, f"FREE[{t}]", value)

@dataclass
class SealedGameBundle:
    geometry: GeometryPackage
    oracle: SealedOracle

def build_geometry_only_package(msg: bytes) -> SealedGameBundle:
    full = compress_single_block_with_trace(msg)
    traces: List[RoundTrace] = full["traces"]
    working_final: List[int] = full["working_final"]

    digest_words = full["digest_words"]
    digest_hex = full["digest_hex"]

    nop_final = NOP["working_final"]
    glass_key = [u32(working_final[i] - nop_final[i]) for i in range(8)]

    t2_carry_bits = "".join(str(rt.t2_carry) for rt in traces)
    t1_hist = {0: 0, 1: 0, 2: 0, 3: 0}
    for rt in traces:
        t1_hist[rt.t1_carry_count] = t1_hist.get(rt.t1_carry_count, 0) + 1

    geom = GeometryPackage(
        digest_hex=digest_hex,
        digest_words=digest_words,
        final_state_word_hw=[hw(x) for x in working_final],
        glass_key_word_hw=[hw(x) for x in glass_key],
        t2_carry_bits=t2_carry_bits,
        t1_carry_histogram=t1_hist,
        delta_hw=[hw(rt.delta) for rt in traces],
        free_hw=[hw(rt.free) for rt in traces],
        nop_delta_hw=[hw(rt.delta) for rt in NOP["traces"]],
    )

    secret_salt = os.urandom(32)
    word_commitments = {rt.t: commitment(secret_salt, f"W[{rt.t}]", rt.Wt) for rt in traces}
    free_commitments = {rt.t: commitment(secret_salt, f"FREE[{rt.t}]", rt.free) for rt in traces}
    schedule_bytes = b"".join(struct.pack(">I", full["W"][t]) for t in range(64))
    schedule_commitment = hashlib.sha256(secret_salt + b"SCHEDULE64" + schedule_bytes).hexdigest()

    oracle = SealedOracle(
        digest_hex=digest_hex,
        secret_salt=secret_salt,
        word_commitments=word_commitments,
        free_commitments=free_commitments,
        schedule_commitment=schedule_commitment,
    )

    # Direct values were used transiently to create the sealed package.
    # They are not returned.
    del full, traces, working_final, nop_final, glass_key, schedule_bytes
    return SealedGameBundle(geometry=geom, oracle=oracle)


In [5]:

@dataclass
class ReverseState:
    t_next: int
    x_next: Tuple[int, int, int, int, int, int, int, int]
    guessed_words: Dict[int, int] = field(default_factory=dict)

@dataclass
class ReverseGame:
    geometry: GeometryPackage
    oracle: SealedOracle
    state: ReverseState

    @staticmethod
    def from_bundle(bundle: SealedGameBundle) -> "ReverseGame":
        digest_words = [int(bundle.geometry.digest_hex[i:i+8], 16) for i in range(0, 64, 8)]
        # For one block: digest_words = H0 + x_64  (mod 2^32)
        x64 = tuple(u32(digest_words[i] - H0[i]) for i in range(8))
        return ReverseGame(
            geometry=bundle.geometry,
            oracle=bundle.oracle,
            state=ReverseState(t_next=63, x_next=x64),
        )

    def done(self) -> bool:
        return self.state.t_next < 0

    def current_round(self) -> int:
        return self.state.t_next

    def _corridor_preview(self) -> Dict[str, int]:
        if self.done():
            raise RuntimeError("Game already complete.")
        t = self.state.t_next
        a1, b1, c1, d1, e1, f1, g1, h1 = self.state.x_next

        # Recover the shift-exposed part of x_t from x_{t+1}
        a = b1
        b = c1
        c = d1
        e = f1
        f = g1
        g = h1

        T2 = u32(Sigma0(a) + maj(a, b, c))
        T1 = u32(a1 - T2)
        d = u32(e1 - T1)

        known_without_hW = u32(Sigma1(e) + ch(e, f, g) + K[t])
        fused_free = u32(T1 - known_without_hW)

        return {
            "t": t,
            "T1": T1,
            "T2": T2,
            "d_prev": d,
            "free_fused": fused_free,
            "delta": u32(a1 - e1),
            "a_hw": hw(a),
            "b_hw": hw(b),
            "c_hw": hw(c),
            "d_hw": hw(d),
            "e_hw": hw(e),
            "f_hw": hw(f),
            "g_hw": hw(g),
            "t1_hw": hw(T1),
            "t2_hw": hw(T2),
            "free_hw_runtime": hw(fused_free),
        }

    def status(self) -> None:
        if self.done():
            print("All 64 round words have been accepted. The block can now be reconstructed.")
            return
        p = self._corridor_preview()
        t = p["t"]
        print("=" * 72)
        print(f"Reverse round target: t = {t}")
        print("Known from hash + current reverse state (geometry summary only):")
        print(f"  hw(a_t)={p['a_hw']}  hw(b_t)={p['b_hw']}  hw(c_t)={p['c_hw']}  hw(d_t)={p['d_hw']}")
        print(f"  hw(e_t)={p['e_hw']}  hw(f_t)={p['f_hw']}  hw(g_t)={p['g_hw']}")
        print(f"  hw(T1_t)={p['t1_hw']}  hw(T2_t)={p['t2_hw']}  hw(Δ_t)={hw(p['delta'])}")
        print("")
        print("Forward-run side clues (sealed geometry package):")
        print(f"  expected T2 carry bit at round {t}: {self.geometry.t2_carry_bits[t]}")
        print(f"  expected Δ hamming weight at round {t}: {self.geometry.delta_hw[t]}")
        print(f"  expected FREE=h_t+W[t] hamming weight at round {t}: {self.geometry.free_hw[t]}")
        print("")
        print("True unresolved fused datum at the wall:")
        print("  FREE_t = h_t + W[t] (mod 2^32)")
        print("But the atomic transport guess needed to step backward exactly is:")
        print(f"  W[{t}]")
        print("=" * 72)

    def step(self, guess_hex: str) -> bool:
        if self.done():
            raise RuntimeError("Game already complete.")
        t = self.state.t_next
        guess = int(guess_hex.strip().lower().replace("0x", ""), 16) & MASK32
        if not self.oracle.check_word(t, guess):
            print(f"Guess for W[{t}] was not accepted.")
            return False

        a1, b1, c1, d1, e1, f1, g1, h1 = self.state.x_next

        a = b1
        b = c1
        c = d1
        e = f1
        f = g1
        g = h1

        T2 = u32(Sigma0(a) + maj(a, b, c))
        T1 = u32(a1 - T2)
        d = u32(e1 - T1)
        h = u32(T1 - (Sigma1(e) + ch(e, f, g) + K[t] + guess))

        self.state.guessed_words[t] = guess
        self.state.x_next = (a, b, c, d, e, f, g, h)
        self.state.t_next -= 1

        print(f"Accepted W[{t}] = 0x{guess:08x}.")
        if not self.done():
            print(f"Moved backward to target round t = {self.state.t_next}.")
        else:
            print("All 64 rounds accepted.")
        return True

    def prompt_step(self) -> bool:
        if self.done():
            print("Game complete.")
            return False
        self.status()
        t = self.current_round()
        guess = input(f"Type W[{t}] as 8-hex digits (or 0x...) : ").strip()
        return self.step(guess)

    def reconstructed_schedule(self) -> List[int]:
        if not self.done():
            raise RuntimeError("Need all 64 guesses before reconstructing the full schedule.")
        return [self.state.guessed_words[t] for t in range(64)]

    def schedule_is_self_consistent(self) -> bool:
        W = self.reconstructed_schedule()
        if len(W) != 64:
            return False
        for t in range(16, 64):
            expect = u32(sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16])
            if W[t] != expect:
                return False
        return True

    def recovered_padded_block(self) -> bytes:
        W = self.reconstructed_schedule()
        if not self.schedule_is_self_consistent():
            raise ValueError("Recovered schedule is not self-consistent.")
        return b"".join(struct.pack(">I", W[i]) for i in range(16))

    def recovered_message(self) -> bytes:
        block = self.recovered_padded_block()
        if len(block) != 64:
            raise ValueError("Recovered block length must be 64.")
        # Standard single-block unpadding
        if block[-8:] is None:
            raise ValueError("Invalid block.")
        bit_len = struct.unpack(">Q", block[-8:])[0]
        msg_len = bit_len // 8
        prefix = block[:msg_len]
        if pad_single_block(prefix) != block:
            raise ValueError("Recovered block does not decode as a valid single-block SHA-256 message.")
        return prefix

    def reveal_message_if_complete(self) -> None:
        if not self.done():
            print("Need all 64 guesses first.")
            return
        print("schedule_is_self_consistent =", self.schedule_is_self_consistent())
        msg = self.recovered_message()
        print("recovered_message_bytes =", msg)
        try:
            print("recovered_message_utf8  =", msg.decode("utf-8"))
        except UnicodeDecodeError:
            print("recovered_message_utf8  = <not valid UTF-8>")



## Create a sealed geometry-only game bundle

Set the message below to anything up to **55 bytes**.
The notebook will compute the full forward pass internally, but only return:

- a geometry package, and
- sealed commitments for answer checking.

No direct `W[t]`, no direct round states, no block bytes are returned.


In [6]:

MESSAGE = b"abc"  # change this to your own single-block message (<= 55 bytes)

bundle = build_geometry_only_package(MESSAGE)

print("Geometry-only package:")
bundle.geometry.summary()

print("\nOracle metadata:")
print("digest matches geometry =", bundle.oracle.digest_hex == bundle.geometry.digest_hex)
print("sealed schedule commitment =", bundle.oracle.schedule_commitment)

# Important: drop the original message variable if you want the notebook session
# to stop carrying it around in the global namespace.
del MESSAGE


Geometry-only package:
digest_hex = ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
final_state_word_hw = [12, 15, 13, 17, 17, 12, 15, 14]
glass_key_word_hw  = [12, 14, 18, 16, 13, 14, 16, 18]
t2_carry_bits      = 1101110010000001110111101111100111101010001110000000100001111100
t1_carry_histogram = {0: 2, 1: 18, 2: 35, 3: 9}
delta_hw[:8]       = [14, 12, 18, 15, 13, 19, 22, 15] ...
free_hw[:8]        = [15, 18, 13, 16, 14, 17, 19, 8] ...
nop_delta_hw[:8]   = [14, 19, 11, 19, 18, 16, 14, 14] ...

Oracle metadata:
digest matches geometry = True
sealed schedule commitment = c493c413cb0f79910c69e6fa4cf9169a08f5db7df3c0790385ceee43fa76993a



## Start the reverse game

The reverse engine gets only:

- the public digest,
- the public SHA constants / IV,
- the sealed geometry package,
- your guesses.

It does **not** use a stored forward trace.


In [7]:

game = ReverseGame.from_bundle(bundle)
game.status()


Reverse round target: t = 63
Known from hash + current reverse state (geometry summary only):
  hw(a_t)=15  hw(b_t)=13  hw(c_t)=17  hw(d_t)=23
  hw(e_t)=12  hw(f_t)=15  hw(g_t)=14
  hw(T1_t)=16  hw(T2_t)=15  hw(Δ_t)=15

Forward-run side clues (sealed geometry package):
  expected T2 carry bit at round 63: 0
  expected Δ hamming weight at round 63: 15
  expected FREE=h_t+W[t] hamming weight at round 63: 15

True unresolved fused datum at the wall:
  FREE_t = h_t + W[t] (mod 2^32)
But the atomic transport guess needed to step backward exactly is:
  W[63]



## Play interactively

Run the next cell repeatedly.  
Each call asks for the exact next transport word `W[t]` needed to move the inverse one step backward.


In [ ]:

game.prompt_step()


Reverse round target: t = 63
Known from hash + current reverse state (geometry summary only):
  hw(a_t)=15  hw(b_t)=13  hw(c_t)=17  hw(d_t)=23
  hw(e_t)=12  hw(f_t)=15  hw(g_t)=14
  hw(T1_t)=16  hw(T2_t)=15  hw(Δ_t)=15

Forward-run side clues (sealed geometry package):
  expected T2 carry bit at round 63: 0
  expected Δ hamming weight at round 63: 15
  expected FREE=h_t+W[t] hamming weight at round 63: 15

True unresolved fused datum at the wall:
  FREE_t = h_t + W[t] (mod 2^32)
But the atomic transport guess needed to step backward exactly is:
  W[63]



## Convenience helpers

- `game.status()` shows the current geometry summary and the next missing datum.
- `game.step("deadbeef")` submits a guess without `input()`.
- `game.reveal_message_if_complete()` reconstructs the message after all 64 guesses are correct.


In [ ]:

# Example helper calls:
# game.status()
# game.step("00000000")
# game.reveal_message_if_complete()



## Auto-demo cell (optional)

This cell demonstrates that the notebook logic works end-to-end by using the original message
*inside a fresh local scope* to reconstruct the correct schedule and feed it back into the game.
It is disabled by default because the point of the notebook is the manual reverse game.


In [ ]:

RUN_AUTODEMO = False

if RUN_AUTODEMO:
    test_msg = b"abc"
    hidden = compress_single_block_with_trace(test_msg)
    demo = ReverseGame.from_bundle(build_geometry_only_package(test_msg))
    for t in range(63, -1, -1):
        demo.step(f"{hidden['W'][t]:08x}")
    demo.reveal_message_if_complete()
else:
    print("Auto-demo disabled.")



## Notes

### What this notebook proves
- Exact backward stepping from the digest works **if** the next `W[t]` is supplied.
- The reverse game can be run with a geometry-only forward package and sealed commitments.
- The geometry package can be kept clean of direct transported values.

### What it does not prove
- It does **not** prove digest-only full preimage inversion.
- It does **not** prove that geometry-only observables uniquely determine every `W[t]`.
- It does **not** eliminate the frontier term; it only isolates it and turns it into a playable reverse game.

### Why this matches the boundary you asked for
The player-visible package contains only side data. The transported values are never exported from the forward pass.
The reverse pass asks you for the missing transport words explicitly, one at a time.
